## Preliminaries and Utils

In [1]:
import numpy as np
import pandas as pd
import warnings
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import  MinMaxScaler, StandardScaler

# from modules.prediction_models import OnlineDecisionTreeRegressor
from modules.prediction_models import OnlineRidgePolynomialRegressor
# from modules.prediction_models import OnlineKNNRegressor

from modules.utils import Metrics, PrintSummary, ShowPlots

In [2]:
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="X has feature names, but PolynomialFeatures was fitted without feature names")
warnings.filterwarnings("ignore", message="X has feature names, but DecisionTreeRegressor was fitted without feature names")
pd.options.mode.chained_assignment = None

In [3]:
summary = PrintSummary()
plots = ShowPlots()
metrics = Metrics()

## Data Loading

In [4]:
data_train = 'datasets/fdata/22_04_train.parquet'
data_test = 'datasets/fdata/22_04_test.parquet'

In [5]:
df_train = pd.read_parquet(data_train, engine="pyarrow").copy()
df_test = pd.read_parquet(data_test, engine="pyarrow").copy()

df_train['Desired QoS'] = df_train['Desired QoS'].astype(int)
df_test['Desired QoS'] = df_test['Desired QoS'].astype(int)

In [6]:
len_test = len(df_test)
len_test

84877

In [7]:
max_runtime_limit = df_test["Requested Time"].max()
max_runtime_limit

259200

## Data Preparation

In [8]:
scaler = MinMaxScaler()
scaler.fit(df_train['Run Time'].values.reshape(-1, 1))

MinMaxScaler()

In [9]:
fs1 = ["User ID", "Requested Number of Nodes", "Requested Number of CPU", "Requested Number of GPU", "Total Requested Memory", "Desired QoS", "Requested Time"]

fs2 = ["User ID", "Requested Number of Nodes", "Requested Number of CPU", "Requested Number of GPU", "Total Requested Memory", "Desired QoS", "Requested Time",
       "Prev Run Time 1", "Prev Run Time 2", "Prev Run Time 3", "Avg Run Time 2", "Avg Run Time 3", "Avg Run Time All"]

fs3 = ["User ID", "Requested Number of Nodes", "Requested Number of CPU", "Requested Number of GPU", "Total Requested Memory", "Desired QoS", "Requested Time",
       "Prev Run Time 1", "Prev Run Time 2", "Prev Run Time 3", "Avg Run Time 2", "Avg Run Time 3", "Avg Run Time All",
       "Prev Requested Nodes 1", "Prev Requested Nodes 2", "Prev Requested Nodes 3", "Avg Requested Nodes 2", "Avg Requested Nodes 3", "Avg Requested Nodes All"]

target_name = "Run Time"

In [10]:
y_train = df_train[target_name]
y_test  = df_test[target_name]

## SET 1

In [11]:
X_train = df_train[fs1]
X_test  = df_test[fs1]

In [12]:
regressor = OnlineRidgePolynomialRegressor(degree=2, batch_size=50, max_time_limit=max_runtime_limit)
regressor.fit(X_train, y_train)

In [13]:
y_pred = []

start = time.time()
# Simulate streaming job arrivals
for i in range(len(X_test)):
    x = X_test.iloc[[i]]
    y = y_test.iloc[i]
    pred = regressor.predict(x)[0]
    # print("Predicted:", pred, "Actual:", y)
    y_pred.append(pred)
    regressor.partial_fit(x, y)   # online update

end = time.time()

y_pred = np.array(y_pred)

In [14]:
metrics.print(scaler, y_test, y_pred, start, end, len_test)

-------------------------------------------
                 METRICS
-------------------------------------------
Inference time: 109.39405584335327
Latency:        0.0012888539397404866
-------------------------------------------
MAE:            5607.323668367167
MAE (hh:mm:ss): 01:33:27
MAE (Scaled):   0.021634528629727017
EA:             0.6503958006703463
MAPE:           5882.0706132404575
-------------------------------------------


In [15]:
# Save the results in the file with all other predictions

df = pd.read_csv("results/fdata/predictions.csv")

df["pred_runtime_rnp_fs1"] = y_pred

df.to_csv("results/fdata/predictions.csv", index=False)

## SET 2

In [11]:
X_train = df_train[fs2]
X_test  = df_test[fs2]

In [12]:
regressor = OnlineRidgePolynomialRegressor(degree=2, batch_size=50, max_time_limit=max_runtime_limit)
regressor.fit(X_train, y_train)

In [13]:
y_pred = []

start = time.time()
# Simulate streaming job arrivals
for i in range(len(X_test)):
    x = X_test.iloc[[i]]
    y = y_test.iloc[i]
    pred = regressor.predict(x)[0]
    # print("Predicted:", pred, "Actual:", y)
    y_pred.append(pred)
    regressor.partial_fit(x, y)   # online update

end = time.time()

y_pred = np.array(y_pred)

In [14]:
metrics.print(scaler, y_test, y_pred, start, end, len_test)

-------------------------------------------
                 METRICS
-------------------------------------------
Inference time: 130.30959486961365
Latency:        0.0015352756915255445
-------------------------------------------
MAE:            8848.352038832663
MAE (hh:mm:ss): 02:27:28
MAE (Scaled):   0.03413926800586712
EA:             0.6157494858466804
MAPE:           7835.410531942621
-------------------------------------------


In [15]:
# Save the results in the file with all other predictions

df = pd.read_csv("results/fdata/predictions.csv")

df["pred_runtime_rnp"] = y_pred

df.to_csv("results/fdata/predictions.csv", index=False)

## SET 3

In [20]:
X_train = df_train[fs3]
X_test  = df_test[fs3]

In [21]:
regressor = OnlineRidgePolynomialRegressor(degree=2, batch_size=50)
regressor.fit(X_train, y_train)

In [22]:
y_pred = []

start = time.time()
# Simulate streaming job arrivals
for i in range(len(X_test)):
    x = X_test.iloc[[i]]
    y = y_test.iloc[i]
    pred = regressor.predict(x)[0]
    # print("Predicted:", pred, "Actual:", y)
    y_pred.append(pred)
    regressor.partial_fit(x, y)   # online update

end = time.time()

y_pred = np.array(y_pred)

In [23]:
metrics.print(scaler, y_test, y_pred, start, end, len_test)

-------------------------------------------
                 METRICS
-------------------------------------------
Inference time: 114.21796727180481
Latency:        0.0013456880812446812
-------------------------------------------
MAE:            6094.282832805118
MAE (hh:mm:ss): 01:41:34
MAE (Scaled):   0.023513345086136173
EA:             0.6194192704840374
MAPE:           7946.3060021386
-------------------------------------------


In [24]:
# Save the results in the file with all other predictions

df = pd.read_csv("results/fdata/predictions.csv")

df["pred_runtime_rnp_fs3"] = y_pred

df.to_csv("results/fdata/predictions.csv", index=False)